## Part 1: Flight Connections Between Cities
### Data Structures and Helper Functions 

### Introduction

In this part of the project, we work with datasets containing information about cities, flights, and airlines.

The goal is to organise this data efficiently so that it can be used later to find the shortest path between two cities.

To achieve this, we use dictionaries to store and access the data quickly instead of repeatedly scanning the dataset.

### Data Structures Used

The following data structures were implemented:

- **names dictionary**  
  Maps a city name to a set of city IDs.  
  This is useful because multiple cities may share the same name.

- **cities dictionary**  
  Stores detailed information about each city:
  - Name
  - Country
  - Set of outgoing flights

- **airlines dictionary**  
  Maps airline IDs to airline names.

These structures improve efficiency and allow quick lookup of information during search.

In [19]:
import csv

# Global data structures
names = {}        # city_name -> set of city_ids
cities = {}       # city_id -> {name, country, flights}
airlines = {}     # airline_id -> airline_name


def load_data(directory):
    """
    Load data from CSV files into memory
    """

    # Load cities
    with open(f"{directory}/cities.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            city_id = row["city_id"]
            name = row["city_name"].lower()

            cities[city_id] = {
                "name": row["city_name"],
                "country": row["country"],
                "flights": set()
            }

            if name not in names:
                names[name] = {city_id}
            else:
                names[name].add(city_id)

    # Load flights
    with open(f"{directory}/flights.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            flight_id = row["flight_id"]
            source = row["source_city_id"]
            destination = row["destination_city_id"]

            cities[source]["flights"].add((flight_id, destination))

    # Load airlines
    with open(f"{directory}/airlines.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            airlines[row["airline_id"]] = row["airline_name"]


def neighbors_for_city(city_id):
    """
    Returns (flight_id, city_id) pairs for cities reachable from given city
    """
    return cities[city_id]["flights"]


def city_id_for_name(name):
    """
    Returns city_id for a city name (handles duplicates)
    """
    city_ids = list(names.get(name.lower(), set()))

    if len(city_ids) == 0:
        return None

    elif len(city_ids) > 1:
        print(f"Which '{name}'?")
        for city_id in city_ids:
            city = cities[city_id]
            print(f"ID: {city_id}, Name: {city['name']}, Country: {city['country']}")
        try:
            chosen_id = input("Enter City ID: ")
            if chosen_id in city_ids:
                return chosen_id
        except ValueError:
            pass
        return None

    else:
        return city_ids[0]

### Key Functions

#### load_data(directory)
This function reads the CSV files and stores the data into dictionaries for efficient access.

#### neighbors_for_city(city_id)
This function returns all cities directly reachable from a given city.  
Each result is a pair: (flight_id, destination_city_id).

This function will later be used by the search algorithm.

#### city_id_for_name(name)
This function converts a city name into its corresponding city ID.  
If multiple cities share the same name, the user is asked to choose the correct one.

In [20]:
# Load dataset
# Load dataset
load_data("../data")

# Example usage
city_id = city_id_for_name("Windhoek")
print("City ID:", city_id)

print("\nNeighbors (Direct Flights):")
for flight in neighbors_for_city(city_id):
    print(flight)

City ID: 1

Neighbors (Direct Flights):
('1005', '3')
('1001', '2')
('1003', '4')


Shortest Path Search

## Lourena 

This section implements the shortest path search between two cities using Breadth-First Search (BFS).  

The cities are treated as nodes in a graph, while flights represent the connections between cities. The goal is to find the shortest route from a starting city to a destination city.

BFS is used because it explores the graph level by level. This means it can find the shortest path in terms of the number of flights, as long as all flights are treated equally.

In [21]:
class Node:
    def __init__(self, state, parent, action):
        self.state = state
        self.parent = parent
        self.action = action

In [22]:
class QueueFrontier:
    def __init__(self):
        self.frontier = []

    def add(self, node):
        self.frontier.append(node)

    def contains_state(self, state):
        return any(node.state == state for node in self.frontier)

    def empty(self):
        return len(self.frontier) == 0

    def remove(self):
        if self.empty():
            raise Exception("Empty frontier")

        node = self.frontier[0]
        self.frontier = self.frontier[1:]
        return node

In [23]:
neighbors_for_city(city_id)

{('1001', '2'), ('1003', '4'), ('1005', '3')}

## Connection with Elsa's Work

This shortest path function depends on the helper function `neighbors_for_city(city)`, which is responsible for returning the neighboring cities that can be reached from the current city.


In [24]:
def shortest_path(start, goal):
    """
    Finds the shortest path between two cities using Breadth-First Search.

    Parameters:
    start: starting city
    goal: destination city

    Returns:
    A list of flights and cities representing the shortest path,
    or None if no path exists.
    """

    start_node = Node(state=start, parent=None, action=None)

    frontier = QueueFrontier()
    frontier.add(start_node)

    explored = set()

    while True:
        if frontier.empty():
            return None

        node = frontier.remove()

        if node.state == goal:
            path = []

            while node.parent is not None:
                path.append((node.action, node.state))
                node = node.parent

            path.reverse()
            return path

        explored.add(node.state)

        for action, state in neighbors_for_city(node.state):
            if state not in explored and not frontier.contains_state(state):
                child = Node(state=state, parent=node, action=action)
                frontier.add(child)


The algorithm then repeats the following steps:

1. Check if the frontier is empty.
   - If it is empty, there is no path between the cities.

2. Remove the first node from the queue.
   - This follows the BFS rule of First In, First Out.

3. Check if the current city is the goal.
   - If it is the goal, the path is reconstructed by following the parent nodes backwards.

4. Add the current city to the explored set.
   - This prevents the same city from being visited again.

5. Expand the current city.
   - The function checks all neighboring cities using `neighbors_for_city()`.

6. Add valid neighboring cities to the frontier.
   - A city is only added if it has not been explored and is not already in the frontier.

In [25]:
# Convert city names to IDs
start = city_id_for_name("Windhoek")
goal = city_id_for_name("Cairo")

# Run BFS
path = shortest_path(start, goal)

# Display result
if path is None:
    print("No path found.")
else:
    print("Shortest path from Windhoek to Cairo:\n")

    print(f"Start: {cities[start]['name']}")

    for flight, city in path:
        print(f"{flight} -> {cities[city]['name']}")

        test_cases = [
    ("Windhoek", "Cairo"),
    ("Windhoek", "Johannesburg"),
    ("Johannesburg", "Cairo")
]

for s, g in test_cases:
    start = city_id_for_name(s)
    goal = city_id_for_name(g)

    path = shortest_path(start, goal)

    print(f"\nRoute from {s} to {g}:")

    if path is None:
        print("No path found.")
    else:
        print(f"Start: {cities[start]['name']}")
        for flight, city in path:
            print(f"{flight} -> {cities[city]['name']}")

Shortest path from Windhoek to Cairo:

Start: Windhoek
1001 -> Johannesburg
1035 -> Nairobi
1017 -> Addis Ababa
1021 -> Cairo

Route from Windhoek to Cairo:
Start: Windhoek
1001 -> Johannesburg
1035 -> Nairobi
1017 -> Addis Ababa
1021 -> Cairo

Route from Windhoek to Johannesburg:
Start: Windhoek
1001 -> Johannesburg

Route from Johannesburg to Cairo:
Start: Johannesburg
1035 -> Nairobi
1017 -> Addis Ababa
1021 -> Cairo


## Results and Conclusion

The BFS algorithm successfully finds the shortest path between cities using the available flight connections.

The queue frontier ensures that cities are explored level by level, which makes BFS suitable for finding the shortest path in terms of the number of flights.

The explored set prevents repeated visits to the same city and avoids unnecessary loops.

This section completes Lourena's assigned task for Part 1: Shortest Path Search.